<a href="https://colab.research.google.com/github/priyag241006/Mule-Shield/blob/main/boi_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xgboost shap imbalanced-learn -q

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/DataSet.csv')
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print("Shape:", df.shape)
print(df['F3924'].value_counts())
print("Mule rate: {:.3f}%".format(100 * df['F3924'].mean()))

In [ ]:
import matplotlib.pyplot as plt

counts = df['F3924'].value_counts()
plt.figure(figsize=(5,4))
plt.bar(['Legitimate (0)', 'Mule (1)'], counts.values, color=['#4CAF50','#E53935'])
plt.title('Target Class Distribution')
plt.ylabel('Number of Accounts')
for i, v in enumerate(counts.values):
    plt.text(i, v + 50, str(v), ha='center')
plt.savefig('class_distribution.png', dpi=150)
plt.show()

In [ ]:
missing_frac = df.isna().mean()
print("Columns 100% missing:", (missing_frac == 1.0).sum())
print("Columns with any missing:", (missing_frac > 0).sum())
print("Columns >60% missing:", (missing_frac > 0.6).sum())

# Drop fully-empty columns
fully_empty = missing_frac[missing_frac == 1.0].index.tolist()
df = df.drop(columns=fully_empty)
print("Shape after dropping empty cols:", df.shape)

In [ ]:
from sklearn.feature_selection import mutual_info_classif

y = df['F3924']
X_raw = df.drop(columns=['F3924'])

# Separate numeric vs categorical
cat_cols = X_raw.select_dtypes(include=['object']).columns.tolist()
num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric:", len(num_cols), "Categorical:", len(cat_cols))

# Keep columns with <=60% missing for scoring (faster, more reliable signal)
keep_cols = [c for c in num_cols if X_raw[c].isna().mean() <= 0.6]
X_num_filled = X_raw[keep_cols].fillna(X_raw[keep_cols].median())

# Mutual info needs no NaNs, sample for speed if needed
mi = mutual_info_classif(X_num_filled, y, random_state=42, n_jobs=-1)
mi_series = pd.Series(mi, index=keep_cols).sort_values(ascending=False)
print(mi_series.head(30))

# Pick top 150 numeric features by mutual info (tune as needed)
top_numeric_features = mi_series.head(150).index.tolist()

In [ ]:
highlighted = ['F115','F321','F527','F531','F670','F1692','F2082','F2122',
               'F2582','F2678','F2737','F2956','F3043','F3836','F3887',
               'F3889','F3891','F3894']

highlighted_present = [f for f in highlighted if f in df.columns]
final_numeric_features = sorted(set(top_numeric_features +
                                     [f for f in highlighted_present if f in num_cols]))

print("Final numeric feature count:", len(final_numeric_features))
print("Categorical features used:", [f for f in highlighted_present if f in cat_cols])

In [ ]:
feat_df = df[final_numeric_features].copy()

# 1. Account age risk flag from F3889 (categorical bucket)
if 'F3889' in df.columns:
    age_risk_map = {'L31D': 3, 'L90D': 2, 'L180D': 1, 'L365D': 0.5, 'G365D': 0}
    feat_df['account_age_risk'] = df['F3889'].map(age_risk_map).fillna(0)

# 2. Occupation vulnerability score from F3891
if 'F3891' in df.columns:
    occ_risk_map = {'student': 3, 'unemployed': 3, 'housewife': 2,
                     'selfemployed': 1.5, 'salaried': 0.5, 'agriculture': 1}
    feat_df['occupation_risk'] = df['F3891'].astype(str).str.lower().map(occ_risk_map).fillna(1)

# 3. Profile completeness score (missing count across ALL original columns)
feat_df['missing_count'] = df.drop(columns=['F3924']).isna().sum(axis=1)

# 4. Transaction amount deviation (z-score) from F3836
if 'F3836' in df.columns:
    amt = df['F3836']
    feat_df['amount_zscore'] = (amt - amt.mean()) / (amt.std() + 1e-9)
    feat_df['F3836'] = amt  # ensure raw value retained

feat_df = feat_df.fillna(feat_df.median(numeric_only=True))
print("Engineered feature matrix shape:", feat_df.shape)

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(feat_df)

iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=42, n_jobs=-1)
iso.fit(X_scaled)

# Lower score_samples = more anomalous; flip sign so higher = more suspicious
feat_df['anomaly_score'] = -iso.score_samples(X_scaled)
print(feat_df['anomaly_score'].describe())

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X = feat_df.copy()
y = df['F3924']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train mule count:", y_train.sum(), "| Test mule count:", y_test.sum())

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print("After SMOTE:", y_train_res.value_counts().to_dict())

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import pandas as pd

# Data preparation (dropping 'leaky_cols')
leaky_cols = ['F3912']  # F2029 looks like genuine signal
feat_df_clean = feat_df.drop(columns=[c for c in leaky_cols if c in feat_df.columns])

X = feat_df_clean.copy()
y = df['F3924']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Model definition and training
model = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
model.fit(X_train_res, y_train_res)

# Predictions and evaluation
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Legitimate','Mule']))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
precision, recall, _ = precision_recall_curve(y_test, y_proba)
print("PR-AUC:", auc(recall, precision))

In [ ]:
# Check correlation of every feature with target
corrs = X.apply(lambda col: col.corr(y) if col.dtype != 'object' else np.nan)
print(corrs.abs().sort_values(ascending=False).head(15))

# Check feature importance from the model itself
importance = pd.Series(model.feature_importances_, index=X_train_res.columns)
print(importance.sort_values(ascending=False).head(10))

# Check if any single feature perfectly separates the classes
for col in X.columns:
    vals_mule = X.loc[y==1, col]
    vals_legit = X.loc[y==0, col]
    if vals_mule.max() < vals_legit.min() or vals_mule.min() > vals_legit.max():
        print(f"PERFECT SEPARATOR FOUND: {col}")

In [ ]:
# Inspect F3912 — is it basically the label in disguise?
print(df.groupby('F3924')['F3912'].describe())
print(df['F3912'].value_counts().head(10))

# Inspect F2029 too
if 'F2029' in df.columns:
    print(df.groupby('F3924')['F2029'].describe())

In [ ]:
leaky_cols = ['F3912']  # only this one — F2029 looks like genuine signal
feat_df_clean = feat_df.drop(columns=[c for c in leaky_cols if c in feat_df.columns])

X = feat_df_clean.copy()
y = df['F3924']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

model = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
model.fit(X_train_res, y_train_res)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Legitimate','Mule']))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
precision, recall, _ = precision_recall_curve(y_test, y_proba)
print("PR-AUC:", auc(recall, precision))

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, precision_score, f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
recalls, precisions, f1s = [], [], []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    sm = SMOTE(random_state=42, k_neighbors=5)
    X_tr_res, y_tr_res = sm.fit_resample(X_tr, y_tr)

    m = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.8,
                       eval_metric='logloss', random_state=42, n_jobs=-1)
    m.fit(X_tr_res, y_tr_res)
    pred = m.predict(X_te)

    r = recall_score(y_te, pred)
    p = precision_score(y_te, pred)
    f = f1_score(y_te, pred)
    recalls.append(r); precisions.append(p); f1s.append(f)
    print(f"Fold {fold+1}: recall={r:.2f}, precision={p:.2f}, f1={f:.2f}")

print(f"\nMean recall: {np.mean(recalls):.2f} (+/- {np.std(recalls):.2f})")
print(f"Mean precision: {np.mean(precisions):.2f} (+/- {np.std(precisions):.2f})")
print(f"Mean F1: {np.mean(f1s):.2f} (+/- {np.std(f1s):.2f})")

In [ ]:
# 1. Confusion matrix (using your clean single-split model)
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Legit','Mule'], yticklabels=['Legit','Mule'])
plt.title('Confusion Matrix — Mule Detection')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2. SHAP top-10 plot
import shap
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, max_display=10, show=False)
plt.savefig('shap_top10.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3. Real per-account SHAP explanation
import pandas as pd
results = X_test.copy()
results['actual'] = y_test.values
results['risk_score'] = (y_proba * 100).round(1)

idx = results['risk_score'].sort_values(ascending=False).index[0]
row_pos = X_test.index.get_loc(idx)
contrib = pd.Series(shap_values[row_pos], index=X_test.columns).sort_values(key=abs, ascending=False)

print(f"Account index: {idx} | Risk Score: {results.loc[idx,'risk_score']} | Actual label: {results.loc[idx,'actual']}")
print(contrib.head(5))

In [ ]:
# 4. Fold comparison bar chart
import numpy as np
folds = ['Fold 1','Fold 2','Fold 3','Fold 4','Fold 5']
recalls_f = [0.56, 0.88, 0.81, 0.81, 0.75]
precisions_f = [0.82, 1.00, 0.76, 0.93, 0.92]
f1s_f = [0.67, 0.94, 0.79, 0.87, 0.83]

x = np.arange(len(folds))
width = 0.25
plt.figure(figsize=(8,5))
plt.bar(x - width, recalls_f, width, label='Recall', color='#E53935')
plt.bar(x, precisions_f, width, label='Precision', color='#1976D2')
plt.bar(x + width, f1s_f, width, label='F1', color='#43A047')
plt.xticks(x, folds)
plt.ylim(0,1.1)
plt.legend()
plt.title('5-Fold Cross-Validation — Mule Class Metrics')
plt.savefig('fold_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(y.value_counts())

In [ ]:
print(importance.head(20))

In [ ]:
print([col for col in X.columns if '3924' in col or '3912' in col])

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values("Importance", ascending=False)

print(importance.head(20))